In [1]:
import pandas as pd
import os
import numpy as np
from shapely.geometry import Point
import geopandas as gpd

import sys
sys.path.append('..')
from helpers import get_county2zone

In [2]:
def get_baseline_load_profiles():
    baseline_load_profiles = {}
    hourly_demand_path = "../data/baseline_load_profiles/processed"
    for fname in os.listdir(hourly_demand_path):
        fpath = os.path.join(hourly_demand_path, fname)
        if os.path.isdir(fpath):
            continue
        df = pd.read_csv(fpath, parse_dates=["timestamp"], index_col="timestamp")
        eia_code = fname.replace('.csv', '')
        baseline_load_profiles[eia_code] = df

    for eia_code, df in baseline_load_profiles.items():
        df = df.loc[df.index.year <= 2023]
        df = pd.concat([df.loc[v] for g, v in df.groupby(df.index.year).groups.items()])
        baseline_load_profiles[eia_code] = df.set_index(df.index.tz_localize('UTC'))

    return baseline_load_profiles

def get_rooftop_pv_cf_meta():
    rooftop_pv_cf_meta = pd.read_csv("../data/distpv_profiles/county_centroid_project_points_conus.csv")
    rooftop_pv_cf_meta['FIPS'] = (
        'p' + rooftop_pv_cf_meta['GEOID'].astype(str).str.zfill(5)
    )
    rooftop_pv_cf_meta_geometry = [
        Point(xy) for xy in zip(rooftop_pv_cf_meta['longitude'], rooftop_pv_cf_meta['latitude'])
    ]
    rooftop_pv_cf_meta = gpd.GeoDataFrame(
        rooftop_pv_cf_meta,
        geometry=rooftop_pv_cf_meta_geometry,
        crs='EPSG:4326'
    )

    return rooftop_pv_cf_meta

def create_gid_county_map(rooftop_pv_cf_meta):
    ## Create mapping between counties and rooftop PV profile GIDs
    county_centroids = gpd.read_file("../data/shapefiles/US_COUNTY_2022")
    county_centroids['geometry'] = county_centroids['geometry'].centroid

    rooftop_pv_cf_meta_matched = rooftop_pv_cf_meta.loc[rooftop_pv_cf_meta.FIPS.isin(county_centroids.rb)]
    
    rooftop_pv_cf_meta_unmatched = rooftop_pv_cf_meta.loc[~rooftop_pv_cf_meta.FIPS.isin(county_centroids.rb)]
    county_centroids_unmatched = county_centroids.loc[~county_centroids.rb.isin(rooftop_pv_cf_meta_matched.FIPS)]
    rooftop_pv_cf_meta_unmatched = (
        gpd.sjoin_nearest(
            county_centroids_unmatched.drop(columns='FIPS').rename(columns={'rb': 'FIPS'})[['FIPS', 'geometry']],
            rooftop_pv_cf_meta_unmatched.drop(columns='FIPS').to_crs(county_centroids.crs),
            how='left'
        )
        .to_crs(rooftop_pv_cf_meta.crs)
        .drop(columns='index_right')
    )
    
    gid_county_map = (
        pd.concat([
            rooftop_pv_cf_meta_matched,
            rooftop_pv_cf_meta_unmatched
        ])
        .groupby('gid')
        ['FIPS']
        .apply(list)
        .explode()
    )

    return gid_county_map

def get_rooftop_pv_cf_profile(sector, gid_county_map):
    # Get rooftop PV CF profiles and normalize
    rooftop_pv_cf = pd.read_csv(f"../data/distpv_profiles/county_level_distpv_profiles_{sector}_2007_2023.csv")
    rooftop_pv_cf["datetime"] = pd.to_datetime(rooftop_pv_cf["Unnamed: 0"].str.split("\'").str[1])
    rooftop_pv_cf = rooftop_pv_cf.set_index("datetime").drop(columns=["Unnamed: 0"])
    rooftop_pv_cf = rooftop_pv_cf / rooftop_pv_cf.max().max()

    # Profiles only have 8760 hours, so leap years are missing the last day of the year
    # To fill in data for the last day, duplicate data from the second to last day
    for year in [2008, 2012, 2016, 2020]:
        df = (
            rooftop_pv_cf.loc[rooftop_pv_cf.index.year == year]
            .copy()
            .tail(24)
        )
        df = df.set_index(df.index.map(lambda x: x.replace(day=31)))
        rooftop_pv_cf = pd.concat([rooftop_pv_cf, df])
    
    rooftop_pv_cf = rooftop_pv_cf.sort_index()
    
    # Add county information
    rooftop_pv_cf.columns = rooftop_pv_cf_meta.gid
    rooftop_pv_cf = (
        rooftop_pv_cf.transpose()
        .merge(gid_county_map, left_index=True, right_index=True)
        .set_index('FIPS')
        .transpose()
    )
    
    rooftop_pv_cf.columns.name = ''
    rooftop_pv_cf.index.names = ['timestamp']
    rooftop_pv_cf.index = pd.to_datetime(rooftop_pv_cf.index)

    return rooftop_pv_cf

def rescale_profile(profile, annual_totals):
    profile_norm = profile.apply(lambda x: x / x.groupby(x.index.year).transform('sum'))
    profile_norm = profile_norm.set_index(profile_norm.index.year, append=True)
    rescaled_profile = (
        profile_norm.mul(annual_totals.T, level=1)
        .fillna(0)
        .droplevel(1)
    )

    return rescaled_profile

In [3]:
# Create name mappings for EIA-930 respondents, subregions, etc.
hourly_rto_demand = pd.read_csv(
    "../data/baseline_load_profiles/raw/2016_hourly_demand_by_rto.csv"
)

hourly_subregion_demand = pd.read_csv(
    "../data/baseline_load_profiles/raw/2016_hourly_demand_by_subregion.csv",
    dtype={'subba': str}
)

def get_subbas(ba):
    subbas = hourly_subregion_demand.loc[hourly_subregion_demand.parent == ba].subba.unique().tolist()
    return subbas

In [4]:
rooftop_pv_cf_meta = get_rooftop_pv_cf_meta()
gid_county_map = create_gid_county_map(rooftop_pv_cf_meta)

rooftop_pv_cf_profiles_by_sector = {}
for sector in ['residential', 'commercial']:
    rooftop_pv_cf_profiles_by_sector[sector] = get_rooftop_pv_cf_profile(sector, gid_county_map)

In [5]:
baseline_load_profiles = get_baseline_load_profiles()
distpv_residential_cf_profiles = (
    rooftop_pv_cf_profiles_by_sector['residential'].loc[(
        rooftop_pv_cf_profiles_by_sector['residential'].index.year.isin(range(2016, 2024))
    )]
)
distpv_commercial_cf_profiles = (
    rooftop_pv_cf_profiles_by_sector['commercial'].loc[(
        rooftop_pv_cf_profiles_by_sector['commercial'].index.year.isin(range(2016, 2024))
    )]
)

In [6]:
retail_sales = pd.read_csv(
    '../data/county_ftm_sales_estimates.csv',
    index_col=['FIPS']
)
retail_sales.columns = [int(col) for col in retail_sales.columns]

direct_use = pd.read_csv('../data/county_direct_use.csv', index_col=['FIPS', 'is_pv'])
direct_use.columns = [int(col) for col in direct_use.columns]
direct_use = direct_use.groupby(direct_use.index.get_level_values('FIPS')).sum()

distpv_residential_consumption = pd.read_csv(
    '../data/county_distpv_residential_consumption.csv',
    index_col=['FIPS']
)
distpv_residential_consumption.columns = [int(col) for col in distpv_residential_consumption.columns]

distpv_non_residential_consumption = pd.read_csv(
    '../data/county_distpv_non_residential_consumption.csv',
    index_col=['FIPS']
)
distpv_non_residential_consumption.columns = [int(col) for col in distpv_non_residential_consumption.columns]

county_load_percentages = pd.read_csv(
    '../data/county_zone_load_percentage.csv',
    index_col=['FIPS', 'EIAcode']
)
county_load_percentages.columns = [int(col) for col in county_load_percentages.columns]

In [7]:
rsdu = retail_sales.add(direct_use, fill_value=0)

for year in range(2016, 2024):
    rsdu = rsdu.merge(
        county_load_percentages[[year]].rename(columns={year: 'weight'}),
        left_index=True,
        right_index=True
    )
    rsdu[year] *= rsdu['weight']
    rsdu = rsdu.drop(columns='weight')

In [8]:
zone_load_profiles = pd.concat({k:v['value'] for k,v in baseline_load_profiles.items()}, axis=1)

In [9]:
rsdu_profiles_by_county = {}
for county in rsdu.index.get_level_values('FIPS').unique():
    county_rsdu_by_zone = rsdu.loc[county]
    zones = list(county_rsdu_by_zone.index)
    county_zone_load_profiles = zone_load_profiles[zones].copy()
    county_rsdu_profile = rescale_profile(
        county_zone_load_profiles, county_rsdu_by_zone
    )
    rsdu_profiles_by_county[county] = county_rsdu_profile.sum(axis=1)

county_rsdu_profiles = pd.concat(rsdu_profiles_by_county, axis=1)
county_distpv_residential_consumption_profiles = rescale_profile(
    distpv_residential_cf_profiles, distpv_residential_consumption
)
county_distpv_non_residential_consumption_profiles = rescale_profile(
    distpv_commercial_cf_profiles, distpv_non_residential_consumption
)

In [10]:
county_load_profiles = (
    county_rsdu_profiles
    .add(county_distpv_residential_consumption_profiles)
    .add(county_distpv_non_residential_consumption_profiles)
)

In [11]:
os.makedirs('../data/outputs', exist_ok=True)
county_load_profiles.to_hdf(
    '../data/outputs/historic_load_hourly_2016_2023_county.h5',
    key='data'
)